## ⚙️ **Libraries Import**

In [ ]:
!pip install protobuf==3.20.3

In [ ]:
!pip install timm transformers huggingface_hub

In [ ]:
import os
import cv2
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils import class_weight

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torchvision.models import (
    efficientnet_v2_s, EfficientNet_V2_S_Weights,
    resnet50, ResNet50_Weights
)

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# 1. Setup Device & Seed
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.benchmark = True
    print(f"🌱 Seed set to {seed}")

SEED = 42
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Device: {device}")

## ⚙️ **Global Config**

In [ ]:
class Config:
    # Paths
    # KAGGLE_PATH = "/kaggle/input/an2dl-second-challenge/an2dl/"
    TRAIN_DIR = '/kaggle/input/cleaned-dataset/cleaned-dataset/train_data'
    TEST_DIR = '/kaggle/input/cleaned-dataset/cleaned-dataset/test_data'
    LABELS_FILE = '/kaggle/input/cleaned-dataset/cleaned-dataset/labels.csv'
    
    # Model Selection: 'phikon', 'uni', 'efficientnet', 'resnet50'
    MODEL_NAME = "efficientnet" 
    
    # Image Settings (Phikon/UNI prefer 224 or 256)
    IMG_SIZE = (224, 224) 

    USE_MASK = True        # Set to False to keep background (don't black it out)
    CROP_TO_ROI = False      # Set to False to use the whole image
    PAD_TO_SQUARE = False    # Set to True to avoid stretching/distorting the image
    
    # Training Hyperparameters
    BATCH_SIZE = 32
    EPOCHS_HEAD = 10       # Head needs less time for Foundation models
    EPOCHS_FINE = 200      # Fine-tuning
    LR_HEAD = 3e-4        # Lower LR for ViTs
    LR_FINE = 1e-5
    PATIENCE = 20
    DROPOUT = 0.4
    
    # HuggingFace Token (Required for UNI, Optional for others)
    # Get yours at https://huggingface.co/settings/tokens
    HF_TOKEN = "YOUR_HUGGINGFACE_TOKEN_HERE" 
    
CONFIG = Config()

## 🧼 **Data cleaning**

In [ ]:
import math

def visualize_rejected_images(rejected_log, max_images=50):
    """
    Plots a grid of rejected images with their reasons.
    Limits display to max_images to prevent crashing the notebook.
    """
    if not rejected_log:
        print("🎉 No images were rejected!")
        return

    n = len(rejected_log)
    display_n = min(n, max_images)
    print(f"👁️ Visualizing {display_n} out of {n} rejected images...")
    
    cols = 5
    rows = math.ceil(display_n / cols)
    
    plt.figure(figsize=(20, 4 * rows))
    
    # Sort by reason so conflicts appear together
    sorted_log = sorted(rejected_log, key=lambda x: x['reason'])[:display_n]
    
    for i, item in enumerate(sorted_log):
        plt.subplot(rows, cols, i + 1)
        
        # Load image
        img = cv2.imread(item['path'])
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.imshow(img)
        else:
            plt.text(0.5, 0.5, "Img Load Error", ha='center')
            
        # Color code the title
        title_color = 'black'
        if "Conflict" in item['reason']:
            title_color = 'red'
        elif "Low Info" in item['reason']:
            title_color = 'gray'
        elif "Redundant" in item['reason']:
            title_color = 'blue'
            
        plt.title(f"{item['filename']}\nLabel: {item['label']}\n{item['reason']}", 
                  fontsize=9, color=title_color, fontweight='bold')
        plt.axis('off')
        
    plt.tight_layout()
    plt.show()

In [ ]:
# --- DATA CLEANING UTILITIES ---

def get_image_fingerprint(img):
    """
    Resizes image to 8x8 grayscale and flattens it.
    Returns a hashable tuple representing the visual content.
    """
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # Resize to tiny grid (fast, ignores minor noise, captures structure)
    img_small = cv2.resize(img_gray, (8, 8), interpolation=cv2.INTER_AREA)
    return img_small

def get_canonical_id(img):
    """
    Generates fingerprints for all 8 geometric transformations of the image.
    Returns the lexicographically smallest fingerprint (Canonical ID).
    This ensures an image and its rotated version get the SAME ID.
    """
    fingerprints = []
    
    # Base image
    base = get_image_fingerprint(img)
    
    # Generate all 8 orientations
    # 0, 90, 180, 270 rotations
    curr = base
    for _ in range(4):
        fingerprints.append(tuple(curr.flatten()))
        # Flip current
        flipped = cv2.flip(curr, 1)
        fingerprints.append(tuple(flipped.flatten()))
        # Rotate 90 deg for next iter
        curr = cv2.rotate(curr, cv2.ROTATE_90_CLOCKWISE)
        
    # The canonical ID is the minimum hash value found among all orientations
    return min(fingerprints)

def clean_dataset_with_logging(folder_path, labels_df, threshold_std=5):
    """
    Scans folder for outliers and duplicates.
    Returns:
      1. valid_filenames: List of files to keep.
      2. rejected_log: List of dicts {'filename', 'reason', 'path'} for visualization.
    """
    print("🧹 Starting Data Cleaning with Visualization Logging...")
    
    img_files = sorted([f for f in os.listdir(folder_path) if f.startswith('img_') and f.endswith('.png')])
    label_map = dict(zip(labels_df['sample_index'], labels_df['label']))
    
    visual_groups = {} 
    rejected_log = [] # Store rejected images here
    valid_filenames = []
    
    # --- PHASE 1: SCAN & DETECT OUTLIERS ---
    print(f"   Scanning {len(img_files)} images...")
    for filename in img_files:
        path = os.path.join(folder_path, filename)
        img = cv2.imread(path)
        if img is None: continue
        
        # 1. Outlier Check
        std_dev = np.std(img)
        if std_dev < threshold_std:
            rejected_log.append({
                'filename': filename,
                'path': path,
                'reason': f"Low Info (Std: {std_dev:.1f})",
                'label': label_map.get(filename, 'N/A')
            })
            continue 
            
        # 2. Fingerprinting for Duplicates
        can_id = get_canonical_id(img)
        if can_id not in visual_groups:
            visual_groups[can_id] = []
        visual_groups[can_id].append(filename)

    # --- PHASE 2: RESOLVE DUPLICATES & CONFLICTS ---
    for can_id, filenames in visual_groups.items():
        # Case A: Unique Image (Keep it)
        if len(filenames) == 1:
            valid_filenames.append(filenames[0])
            continue
            
        # Case B: Duplicates Found
        current_labels = [label_map.get(f, 'Unknown') for f in filenames]
        unique_labels = set(current_labels)
        
        if len(unique_labels) == 1:
            # SAME LABELS: Keep first, reject others as "Redundant"
            keep = filenames[0]
            valid_filenames.append(keep)
            
            for drop_file in filenames[1:]:
                rejected_log.append({
                    'filename': drop_file,
                    'path': os.path.join(folder_path, drop_file),
                    'reason': f"Redundant Copy (Kept {keep})",
                    'label': current_labels[0]
                })
        else:
            # DIFFERENT LABELS: Conflict! Reject ALL.
            conflict_str = " vs ".join([str(l) for l in unique_labels])
            for f, l in zip(filenames, current_labels):
                rejected_log.append({
                    'filename': f,
                    'path': os.path.join(folder_path, f),
                    'reason': f"Conflict: {conflict_str}",
                    'label': l
                })

    print(f"✅ retained {len(valid_filenames)} images.")
    print(f"❌ Rejected {len(rejected_log)} images.")
    
    return valid_filenames, rejected_log

## ⏳ **Data Loading**

In [ ]:
snots = ['img_0102.png',
'img_0108.png',
'img_0109.png',
'img_0152.png',
'img_0153.png',
'img_0168.png',
'img_0176.png',
'img_0182.png',
'img_0223.png',
'img_0232.png',
'img_0239.png',
'img_0256.png',
'img_0270.png',
'img_0276.png',
'img_0277.png',
'img_0282.png',
'img_0304.png',
'img_0318.png',
'img_0323.png',
'img_0369.png',
'img_0384.png',
'img_0390.png',
'img_0411.png',
'img_0428.png',
'img_0436.png',
'img_0442.png',
'img_0448.png',
'img_0451.png',
'img_0455.png',
'img_0471.png',
'img_0480.png',
'img_0489.png',
'img_0493.png',
'img_0503.png',
'img_0511.png',
'img_0512.png',
'img_0518.png',
'img_0521.png',
'img_0526.png',
'img_0554.png',
'img_0559.png',
'img_0572.png',
'img_0592.png',
'img_0597.png',
'img_0600.png',
'img_0612.png',
'img_0650.png',
'img_0652.png',
'img_0687.png',
'img_0714.png',
'img_0796.png',
'img_0804.png',
'img_0817.png',
'img_0822.png',
'img_0829.png',
'img_0866.png',
'img_0892.png',
'img_0893.png',
'img_0898.png',
'img_0906.png',
'img_0907.png',
'img_0910.png',
'img_0913.png',
'img_0918.png',
'img_0924.png',
'img_0935.png',
'img_0941.png',
'img_0944.png',
'img_0963.png',
'img_0973.png',
'img_0982.png',
'img_0992.png',
'img_1004.png',
'img_1009.png',
'img_1021.png',
'img_1023.png',
'img_1039.png',
'img_1041.png',
'img_1086.png',
'img_1108.png',
'img_1109.png',
'img_1145.png',
'img_1202.png',
'img_1218.png',
'img_1223.png',
'img_1232.png',
'img_1267.png',
'img_1271.png',
'img_1300.png',
'img_1312.png',
'img_1320.png',
'img_1326.png',
'img_1327.png',
'img_1329.png',
'img_1332.png',
'img_1334.png',
'img_1338.png',
'img_1362.png',
'img_1388.png',
'img_1392.png']

In [ ]:
def pad_to_square(img, mask=None):
    """
    Pads an image to make it square without stretching the content.
    Crucial for pathology to preserve cell shape.
    """
    h, w = img.shape[:2]
    if h == w:
        return img, mask
    
    diff = abs(h - w)
    pad1 = diff // 2
    pad2 = diff - pad1
    
    if h > w: # Pad width
        padding = ((0, 0), (pad1, pad2), (0, 0)) # (top, bot), (left, right), (chan)
        if mask is not None:
            mask_padding = ((0, 0), (pad1, pad2))
    else: # Pad height
        padding = ((pad1, pad2), (0, 0), (0, 0))
        if mask is not None:
            mask_padding = ((pad1, pad2), (0, 0))
            
    img = np.pad(img, padding, mode='constant', constant_values=255) # Pad with White (255) for medical
    if mask is not None:
        mask = np.pad(mask, mask_padding, mode='constant', constant_values=0)
        
    return img, mask

In [ ]:
from PIL import Image

def extract_all_rois(img_path, mask_path=None, label=None, img_name=None,
                     min_area=500, target_size=(224,224), resize=True, debug=False):
    """
    Extracts ROIs using the mask to find bounding boxes.
    RETURNS a list of dictionaries:
       { "roi": ROI_image, "label": label, "img_name": img_name, "bbox": (...) }
    """

    rgb = np.array(Image.open(img_path).convert("RGB")).astype(np.float32)

    # Load mask
    if mask_path is not None and os.path.exists(mask_path):
        mask = np.array(Image.open(mask_path).convert("L"))
        mask = (mask > 0).astype(np.uint8)
        if debug: print("Using external mask:", mask_path)
    else:
        rgba = np.array(Image.open(img_path).convert("RGBA"))
        mask = (rgba[..., 3] > 0).astype(np.uint8)
        if debug: print("Fallback: using alpha mask")

    # Debug image + mask
    if debug:
        plt.figure(figsize=(10,4))
        plt.subplot(1,2,1); plt.imshow(rgb.astype(np.uint8)); plt.title("Image"); plt.axis("off")
        plt.subplot(1,2,2); plt.imshow(mask, cmap="gray"); plt.title("Mask"); plt.axis("off")
        plt.show()

    # Connected components
    num_labels, labels = cv2.connectedComponents(mask)
    roi_entries = []
    margin = 32
    H, W = mask.shape

    if debug:
        print(f"Connected components found: {num_labels-1}")

    # Loop through components
    for comp in range(1, num_labels):

        ys, xs = np.where(labels == comp)
        size = len(xs)
        if size == 0:
            continue

        x_min, x_max = xs.min(), xs.max()
        y_min, y_max = ys.min(), ys.max()

        # Skip if too small
        if size < min_area:
            if debug:
                print(f"Rejected comp {comp} (size {size})")
            continue

        # Expand bbox
        x_min = max(0, x_min - margin)
        y_min = max(0, y_min - margin)
        x_max = min(W - 1, x_max + margin)
        y_max = min(H - 1, y_max + margin)


        # Extract RAW ROI (no mask applied)
        roi_raw = rgb[y_min:y_max, x_min:x_max].astype(np.float32) / 255.0  

        # Optional resize (LANCZOS for sharpness)
        if resize:
            roi_uint8 = (roi_raw * 255).astype(np.uint8)
            roi_pil = Image.fromarray(roi_uint8)
            roi_pil = roi_pil.resize(target_size, Image.LANCZOS)
            roi = np.array(roi_pil).astype(np.float32) / 255.0
        else:
            roi = roi_raw

        # DEBUG SHOW
        if debug:
            print(f"ACCEPTED ROI {comp}: bbox=({x_min},{y_min})–({x_max},{y_max}), size={size}")
            plt.figure(figsize=(4,4))
            plt.imshow(np.clip(roi,0,1))
            plt.title(f"ROI {comp} (label={label})")
            plt.axis("off")
            plt.show()

        # ---------------------------------------------------------
        # ADD ROI + LABEL + METADATA TO OUTPUT
        # ---------------------------------------------------------
        roi_entries.append({
            "roi": roi,
            "label": label,
            "img_name": img_name if img_name is not None else os.path.basename(img_path),
            "bbox": (x_min, y_min, x_max, y_max),
        })

    return roi_entries

In [ ]:
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder

def build_dataset_from_rois(df, data_dir, config):
    """
    Iterates through the filtered dataframe, extracts ROIs,
    and returns a list of dictionaries containing ROI data and metadata.
    """
    all_samples = []
    
    print(f"🚀 Processing {len(df)} files using extract_all_rois...")

    # Iterate through the DataFrame
    for _, row in tqdm(df.iterrows(), total=len(df)):
        img_name = row["sample_index"]
        raw_label = row["label"]  # Keep as string for now
        
        img_path = os.path.join(data_dir, img_name)
        # Ensure mask path logic matches your directory structure
        mask_path = img_path.replace("img_", "mask_") 
        
        # Call your extraction function
        rois = extract_all_rois(
            img_path=img_path,
            mask_path=mask_path,
            label=raw_label,
            img_name=img_name,
            resize=True
        )
        
        # Extend the master list
        if rois:
            all_samples.extend(rois)

    print(f"✅ Extraction complete. Generated {len(all_samples)} samples from {len(df)} images.")
    return all_samples

In [ ]:
# A. Load Labels
labels_df = pd.read_csv(CONFIG.LABELS_FILE)

# B. Run Cleaning 
# (Returns a list of valid filenames 'clean_files_list')
clean_files_list, rejected_log = clean_dataset_with_logging(CONFIG.TRAIN_DIR, labels_df) 

clean_files_list = [x for x in clean_files_list if x not in snots]
# C. Filter DataFrame
# Critical: Only keep rows where the file actually exists and is valid
valid_df = labels_df[labels_df['sample_index'].isin(clean_files_list)].copy()

In [ ]:
# D. Build Dataset
# This replaces the manual loop you had at the bottom
roi_dataset = build_dataset_from_rois(valid_df, CONFIG.TRAIN_DIR, CONFIG)

# -------------------------------------------------------
# 3. Transform and Encode
# -------------------------------------------------------

if len(roi_dataset) == 0:
    raise ValueError("❌ No ROIs were extracted! Check paths, mask settings, or image integrity.")

# A. Create X_raw (Stack the image arrays)
# roi_dataset is a list of dicts. We extract 'roi' from each.
X_list = [item['roi'] for item in roi_dataset]
X_raw = np.stack(X_list) 

# B. Create Names List (for tracking/debugging which ROI came from which image)
train_fnames = [item['img_name'] for item in roi_dataset]

# C. Create Labels
# Extract the raw text label from the dicts
y_raw = [item['label'] for item in roi_dataset]

# D. Encode Labels (String -> Integer)
# This replaces the manual label_to_idx dictionary
le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)

# -------------------------------------------------------
# 4. Final Output Check
# -------------------------------------------------------
print("-" * 30)
print(f"✅ Final X Shape: {X_raw.shape}") # Should be (N, Height, Width, Channels)
print(f"✅ Final y Shape: {y_encoded.shape}")
print(f"✅ Class Mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print("-" * 30)

# OPTIONAL: Save the encoder for later inference
# import joblib
# joblib.dump(le, 'label_encoder.joblib')

##  📄 **Data Preprocessing**

In [ ]:
class MedicalDataset(Dataset):
    def __init__(self, images, labels=None, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx]
        
        if self.transform:
            image = self.transform(image)
            
        if self.labels is not None:
            label = torch.tensor(self.labels[idx], dtype=torch.long)
            return image, label
        return image

# --- AUGMENTATION WITH AUTO-AUGMENT ---
def get_transforms():
    train_tf = v2.Compose([
        v2.ToImage(),
        # ⚡ Auto Augmentation: Automatically learns best transforms
        v2.RandAugment(num_ops=2, magnitude=3), 
        
        v2.ToDtype(torch.float32, scale=True), 
        v2.RandomResizedCrop(size=CONFIG.IMG_SIZE, scale=(0.85, 1.0), antialias=True),
        v2.RandomHorizontalFlip(p=0.5),
        v2.RandomVerticalFlip(p=0.5),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_tf = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return train_tf, val_tf

# Split Data
X_train, X_val, y_train, y_val = train_test_split(
    X_raw, y_encoded, test_size=0.15, stratify=y_encoded, random_state=SEED
)

# Create Datasets
train_tf, val_tf = get_transforms()
train_ds = MedicalDataset(X_train, y_train, transform=train_tf)
val_ds = MedicalDataset(X_val, y_val, transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=CONFIG.BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=CONFIG.BATCH_SIZE, shuffle=False, num_workers=2)

##  🛠️ **Model building**

In [ ]:
import timm
from transformers import AutoImageProcessor, ViTModel

def build_model(model_name, num_classes, dropout_rate=0.3):
    print(f"🏗️ Building model: {model_name.upper()}")
    
    # --- OPTION A: OWKIN PHIKON (Open Access SOTA) ---
    if model_name == "phikon":
        # Phikon is a ViT-Base. We load the body and attach a custom head.
        # Note: We use the HuggingFace 'transformers' library here
        print("   Loading Owkin/Phikon (ViT-Base)...")
        base_model = ViTModel.from_pretrained("owkin/phikon", add_pooling_layer=False)
        
        class PhikonClassifier(nn.Module):
            def __init__(self, base, n_classes, drop):
                super().__init__()
                self.base = base
                self.norm = nn.LayerNorm(768) # Phikon output dim is 768
                self.head = nn.Sequential(
                    nn.Dropout(drop),
                    nn.Linear(768, 512),
                    nn.GELU(),
                    nn.Dropout(drop),
                    nn.Linear(512, n_classes)
                )
            def forward(self, x):
                # Phikon expects pixel values [0,1] or normalized? 
                # Ideally use their processor, but standard normalization works well.
                out = self.base(x).last_hidden_state[:, 0] # Take [CLS] token
                out = self.norm(out)
                return self.head(out)
                
        model = PhikonClassifier(base_model, num_classes, dropout_rate)

    # --- OPTION B: MAHMOOD LAB UNI (Restricted Access SOTA) ---
    elif model_name == "uni":
        print("   Loading MahmoodLab/UNI (ViT-Large)...")
        print("   ⚠️ Ensure you have accepted the license on HuggingFace and set CONFIG.HF_TOKEN")
        
        # UNI is often loaded via timm with specific register commands
        # Depending on the specific repo version, loading might vary. 
        # This is the standard timm approach for a ViT-Large:
        model = timm.create_model(
            "vit_large_patch16_224", 
            pretrained=False, 
            num_classes=num_classes,
            drop_rate=dropout_rate
        )
        
        # Load the custom weights manually or via hub if you have access
        # If using the 'timm' integration for UNI (requires login):
        try:
            # Attempt to load from HF Hub directly (requires 'huggingface-cli login' or token)
            model = timm.create_model(
                "hf_hub:MahmoodLab/UNI", 
                pretrained=True, 
                num_classes=num_classes,
                drop_rate=dropout_rate
            )
        except Exception as e:
            print(f"   ❌ Could not load UNI directly: {e}")
            print("   Fallback: Using standard ImageNet ViT-Large (Not Pathology Pretrained)")
            model = timm.create_model("vit_large_patch16_224", pretrained=True, num_classes=num_classes)

    # --- OPTION C: EFFICIENTNET V2 (Standard Baseline) ---
    elif model_name == "efficientnet":
        weights = EfficientNet_V2_S_Weights.DEFAULT
        model = efficientnet_v2_s(weights=weights)
        input_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(input_features, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(), # Swish is standard for EfficientNet
            nn.Dropout(p=dropout_rate),
            nn.Linear(256, num_classes)
        )

    # --- OPTION D: RESNET50 (Classic Baseline) ---
    elif model_name == "resnet50":
        weights = ResNet50_Weights.DEFAULT
        model = resnet50(weights=weights)
        input_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(input_features, num_classes)
        )
        
    else:
        raise ValueError(f"Model {model_name} not found.")
        
    return model.to(device)

def set_parameter_requires_grad(model, feature_extracting):
    # Helper to freeze/unfreeze
    if feature_extracting:
        # 1. Freeze EVERYTHING first
        for param in model.parameters():
            param.requires_grad = False
            
        # 2. Unfreeze specific Classification Heads based on architecture
        if hasattr(model, 'classifier'): # EfficientNet
            for param in model.classifier.parameters(): 
                param.requires_grad = True
                
        elif hasattr(model, 'fc'): # ResNet
            for param in model.fc.parameters(): 
                param.requires_grad = True
                
        elif hasattr(model, 'head'): # Phikon (ViT) & UNI
            for param in model.head.parameters(): 
                param.requires_grad = True
                
        # 3. Also unfreeze the LayerNorm if it exists (Crucial for ViTs)
        if hasattr(model, 'norm'):
            for param in model.norm.parameters():
                param.requires_grad = True
                
        print("✅ Model frozen. Only head (and norm) are trainable.")
        
    else:
        for param in model.parameters():
            param.requires_grad = True
        print("✅ Model unfrozen. All layers are trainable.")

## 🧠 **Model Training**

In [ ]:
from sklearn.metrics import f1_score

class Trainer:
    def __init__(self, model, criterion, optimizer, scaler, device,
                 monitor="val_loss", mode="min", min_delta=0.0):
        """
        monitor: 'val_loss' o 'val_f1'
        mode: 'min' se vuoi minimizzare (es. loss), 'max' se vuoi massimizzare (es. F1)
        min_delta: miglioramento minimo richiesto per resettare la pazienza
        """
        self.model = model
        self.criterion = criterion
        self.optimizer = optimizer
        self.scaler = scaler
        self.device = device

        self.monitor = monitor
        self.mode = mode
        self.min_delta = min_delta

        self.history = {
            'train_loss': [],
            'train_f1':   [],
            'val_loss':   [],
            'val_f1':     []
        }

        self.best_val_f1 = 0.0  # utile per report a fine training

    def train_epoch(self, loader):
        self.model.train()
        total_loss = 0.0
        all_preds, all_labels = [], []

        for imgs, lbls in loader:
            imgs, lbls = imgs.to(self.device), lbls.to(self.device)
            self.optimizer.zero_grad()

            use_amp = (self.device.type == "cuda")

            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = self.model(imgs)
                loss = self.criterion(logits, lbls)

            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()

            total_loss += loss.item() * imgs.size(0)
            all_preds.extend(logits.argmax(dim=1).detach().cpu().numpy())
            all_labels.extend(lbls.detach().cpu().numpy())

        avg_loss = total_loss / len(loader.dataset)
        avg_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
        return avg_loss, avg_f1

    def validate_epoch(self, loader):
        self.model.eval()
        total_loss = 0.0
        all_preds, all_labels = [], []

        use_amp = (self.device.type == "cuda")

        with torch.no_grad():
            for imgs, lbls in loader:
                imgs, lbls = imgs.to(self.device), lbls.to(self.device)

                with torch.cuda.amp.autocast(enabled=use_amp):
                    logits = self.model(imgs)
                    loss = self.criterion(logits, lbls)

                total_loss += loss.item() * imgs.size(0)
                all_preds.extend(logits.argmax(dim=1).detach().cpu().numpy())
                all_labels.extend(lbls.detach().cpu().numpy())

        avg_loss = total_loss / len(loader.dataset)
        avg_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
        return avg_loss, avg_f1

    def _is_improvement(self, current, best):
        """Ritorna True se current è un miglioramento rispetto a best, dato mode e min_delta."""
        if self.mode == "min":
            return current < best - self.min_delta
        elif self.mode == "max":
            return current > best + self.min_delta
        else:
            raise ValueError("mode must be 'min' or 'max'")

    def fit(self, train_loader, val_loader, epochs, patience, save_name):
        # Inizializza best_score in base a cosa monitoriamo
        if self.mode == "min":
            best_score = float('inf')
        else:
            best_score = -float('inf')

        patience_counter = 0

        for epoch in range(epochs):
            t_loss, t_f1 = self.train_epoch(train_loader)
            v_loss, v_f1 = self.validate_epoch(val_loader)

            # Log completo
            self.history['train_loss'].append(t_loss)
            self.history['train_f1'].append(t_f1)
            self.history['val_loss'].append(v_loss)
            self.history['val_f1'].append(v_f1)

            # Per comodo tenerne anche uno a parte
            self.best_val_f1 = max(self.best_val_f1, v_f1)

            print(
                f"Epoch {epoch+1}/{epochs} | "
                f"T_Loss: {t_loss:.4f} T_F1: {t_f1:.4f} | "
                f"V_Loss: {v_loss:.4f} V_F1: {v_f1:.4f}"
            )

            # Scegli la metrica da monitorare
            current_metric = v_loss if self.monitor == "val_loss" else v_f1

            if self._is_improvement(current_metric, best_score):
                best_score = current_metric
                patience_counter = 0
                torch.save(self.model.state_dict(), f"{save_name}.pt")
                # print(f"  ✅ New best {self.monitor}: {best_score:.4f}")
            else:
                patience_counter += 1
                # print(f"  ⏳ No improvement. Patience: {patience_counter}/{patience}")
                if patience_counter >= patience:
                    print(f"🛑 Early stopping at epoch {epoch+1}")
                    break

        print(f"Restoring best model ({self.monitor} best = {best_score:.4f})")
        self.model.load_state_dict(torch.load(f"{save_name}.pt"))


In [ ]:
# 1. Initialize Model
num_classes = len(le.classes_)
model = build_model(CONFIG.MODEL_NAME, num_classes, CONFIG.DROPOUT)

# 2. Setup Tools
class_weights = class_weight.compute_class_weight(
    "balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
scaler = torch.cuda.amp.GradScaler()

# 3. STAGE 1: Train Head Only
print("\n🔥 STAGE 1: Training Head...")
set_parameter_requires_grad(model, feature_extracting=True)

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG.LR_HEAD
)

trainer = Trainer(
    model, 
    criterion, 
    optimizer, 
    scaler, 
    device,
    monitor="val_loss", 
    mode="min",         
    min_delta=0.001
)

trainer.fit(
    train_loader, val_loader,
    epochs=CONFIG.EPOCHS_HEAD,
    patience=10,
    save_name="stage1_model"
)

# 4. STAGE 2: Fine-Tuning Backbone
print("\n🔥 STAGE 2: Fine Tuning Backbone...")
set_parameter_requires_grad(model, feature_extracting=False)

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG.LR_FINE,
    weight_decay=1e-4
)

trainer.optimizer = optimizer

trainer.fit(
    train_loader,
    val_loader,
    epochs=CONFIG.EPOCHS_FINE,
    patience=CONFIG.PATIENCE,
    save_name="best_model"
)

# 5. Plot Results
plt.figure(figsize=(10,4))
plt.plot(trainer.history['train_loss'], label='Train Loss')
plt.plot(trainer.history['val_f1'], label='Val F1')
plt.legend()
plt.show()


## 💫 **Confusion Matrix**

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

def generate_confusion_matrix(model, loader, device, class_names):
    """
    Runs the model on the loader, generates predictions, and plots the matrix.
    """
    model.eval()
    all_preds = []
    all_labels = []
    
    print("📊 Generating Confusion Matrix...")
    
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            lbls = lbls.to(device)
            
            # Use AutoCast for consistent performance
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(imgs)
            
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())
            
    # Calculate Matrix
    cm = confusion_matrix(all_labels, all_preds)
    
    # Calculate Normalized Matrix (Recall per class)
    # epsilon added to prevent division by zero
    cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-10)
    
    # Plotting
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    # Raw Counts
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names, 
                ax=axes[0], square=True)
    axes[0].set_title('Confusion Matrix (Raw Counts)', fontweight='bold')
    axes[0].set_ylabel('True Label')
    axes[0].set_xlabel('Predicted Label')
    
    # Normalized
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens', 
                xticklabels=class_names, yticklabels=class_names, 
                ax=axes[1], square=True)
    axes[1].set_title('Confusion Matrix (Normalized %)', fontweight='bold')
    axes[1].set_ylabel('True Label')
    axes[1].set_xlabel('Predicted Label')
    
    plt.tight_layout()
    plt.show()
    
    return all_labels, all_preds

In [ ]:
print("\n🧐 Evaluating Best Model on Validation Set...")

# 1. Ensure we have the best model loaded (Trainer does this automatically, but just to be safe)
model.load_state_dict(torch.load("best_model.pt"))

# 2. Run the Visualization
# le.classes_ contains the names ['Dysplasia', 'Normal', 'Cancer'] etc.
true_labels, pred_labels = generate_confusion_matrix(model, val_loader, device, le.classes_)

# 3. Print a detailed Text Report as well
print("\n📑 Classification Report:")
print(classification_report(true_labels, pred_labels, target_names=le.classes_))

## 🕹️ **Use the Model - Make Inference**

In [ ]:
print("\n🔎 Processing Test Data...")

# 1. Reuse the unified loader with file_list=None to scan the whole folder
X_test, test_filenames = load_images_with_smart_crop(
    CONFIG.TEST_DIR, 
    file_list=None,  # None triggers "Load All" mode
    target_size=CONFIG.IMG_SIZE
)

if len(X_test) > 0:
    # 2. Create Loader (Use Val transforms for normalization)
    test_ds = MedicalDataset(X_test, labels=None, transform=val_tf)
    test_loader = DataLoader(test_ds, batch_size=CONFIG.BATCH_SIZE, shuffle=False)

    # 3. Predict
    print("🔮 Running Inference...")
    model.eval()
    all_preds = []

    with torch.no_grad():
        for imgs in test_loader:
            imgs = imgs.to(device)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(imgs)
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())

    # 4. Decode & Save
    predicted_labels = le.inverse_transform(all_preds)

    submission = pd.DataFrame({
        'sample_index': test_filenames,
        'label': predicted_labels
    })

    submission.to_csv("submission.csv", index=False)
    print("🏆 Submission saved successfully!")
    print(submission.head())
else:
    print("⚠️ No images found in Test Directory.")

In [ ]:
print("\n🔎 Processing Test Data...")

test_rois = []
test_image_ids = []   # track which ROI belongs to which WSI

# 1. Loop through files
for fname in sorted(os.listdir(CONFIG.TEST_DIR)):
    # Basic extension check
    if not fname.lower().endswith((".png",".jpg",".jpeg",".tif",".bmp")):
        continue
    
    # --- 🟢 NEW ADDITION: Skip mask files immediately ---
    # This prevents them from entering the pipeline entirely.
    if "mask_" in fname:
        continue
    # ----------------------------------------------------

    img_path = os.path.join(CONFIG.TEST_DIR, fname)
    # Assumes your files are named like 'img_01.png' -> 'mask_01.png'
    mask_path = img_path.replace("img_", "mask_") 

    # 2. Extract ROIs
    rois = extract_all_rois(
        img_path=img_path,
        mask_path=mask_path,
        label=None,
        img_name=fname,
        resize=True
    )

    # 3. Fallback if no ROI found
    if len(rois) == 0:
        print(f"⚠️ Warning: No ROI found for {fname}. Using center crop fallback.")
        
        fallback = Image.open(img_path).convert("RGB")
        fallback = fallback.resize((224,224), Image.LANCZOS)
        fallback = np.array(fallback).astype(np.float32)/255.0
        
        rois = [{"roi": fallback, "img_name": fname, "label": None}]

    for r in rois:
        test_rois.append(r["roi"])
        test_image_ids.append(fname)

# ... The rest of your dataset creation and inference code remains exactly the same ...
test_ds = MedicalDataset(
    images=np.stack(test_rois),
    labels=None,
    transform=val_tf
)

test_loader = DataLoader(test_ds, batch_size=CONFIG.BATCH_SIZE, shuffle=False)

print("🔮 Running Inference...")
model.eval()

all_probs = []

with torch.no_grad():
    for imgs in test_loader:
        imgs = imgs.to(device)
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logits = model(imgs)
            probs = torch.softmax(logits, dim=1)
        all_probs.append(probs.cpu().numpy())

all_probs = np.vstack(all_probs)

from collections import defaultdict
import numpy as np

image_to_probs = defaultdict(list)

for prob, img_id in zip(all_probs, test_image_ids):
    image_to_probs[img_id].append(prob)

final_preds = {}
for img_id, prob_list in image_to_probs.items():
    mean_prob = np.mean(prob_list, axis=0)
    final_class_idx = np.argmax(mean_prob)
    final_preds[img_id] = final_class_idx

predicted_labels = {
    img_id: le.inverse_transform([cls_idx])[0]
    for img_id, cls_idx in final_preds.items()
}

submission = pd.DataFrame({
    "sample_index": list(predicted_labels.keys()),
    "label": list(predicted_labels.values())
})

submission.to_csv("submission.csv", index=False)
print("🏆 Submission saved successfully!")


In [ ]:
# ... previous code ...

submission = pd.DataFrame({
    "sample_index": list(predicted_labels.keys()),
    "label": list(predicted_labels.values())
})

# Filter out the masks right before saving
submission = submission[~submission['sample_index'].str.contains("mask_")]

submission.to_csv("FInalsubmission.csv", index=False)
print("🏆 Submission saved successfully!")